In [ ]:
import torch
import numpy as np
from environment import CliffWalk   # or InfiniteWalk
from episodes import EpisodeCollection
from actor import collect_episodes_actor
from train_chunk import train_with_chunks
from nn_models import BeliefRNN, RewardReadout, NextLatentPredictor, ObsReadout, ValueReadout, ActorReadout, QReadout



In [ ]:
# --------------------------
# 1. Hyperparameters
# --------------------------

RNN_HIDDEN = 64
INITIAL_CHUNKS = 1
EPISODES_PER_CHUNK = 100
NUM_NEW_CHUNKS = 100

GAMMA = 0.9

# Training step counts per chunk
ACTOR_STEPS  = 10
CRITIC_STEPS = 50
WORLD_STEPS  = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# --------------------------
# 2. Initialize Environment
# --------------------------
env = CliffWalk(
    n = 2, 
    m = 5, 
    self_transition_prob=0.1,
    gamma=GAMMA,
    generic_reward=-1.0,
    cliff_reward = -10.0,
    target_reward=10.0,
)  



# --------------------------
# 3. Collect Initial Chunks Using a Random Policy
# --------------------------
# Use a dummy policy for initial data collection (uniform over actions)
def random_policy_wrapper(env):
    class RandomPolicy:
        def reset(self): pass
        def __call__(self, obs, prev_action):
            action = np.random.choice(env.action_space)
            return action, 0.0
    return RandomPolicy()

initial_chunks = []
for _ in range(INITIAL_CHUNKS):
    eps = collect_episodes_actor(env, actor_policy=random_policy_wrapper(env), num_episodes=500)
    EC = EpisodeCollection(eps)
    initial_chunks.append(EC)

In [ ]:
# --------------------------
# 4. Initialize Models
# --------------------------
input_dim = initial_chunks[0].H      # history dimension = O + A
obs_dim   = initial_chunks[0].O      # observation dimension
num_actions = initial_chunks[0].A    # action dimension

belief_model = BeliefRNN(input_dim=input_dim, latent_dim=RNN_HIDDEN)
value_model  = ValueReadout(RNN_HIDDEN)
q_model      = QReadout(RNN_HIDDEN, 4)
reward_model = RewardReadout(RNN_HIDDEN)
pred_model   = NextLatentPredictor(RNN_HIDDEN)
obs_model    = ObsReadout(RNN_HIDDEN, obs_dim)
actor_model  = ActorReadout(latent_dim=RNN_HIDDEN, num_actions=num_actions, hidden_dim=64)

# Move to device
for m in (belief_model, value_model, q_model, reward_model, pred_model, obs_model, actor_model):
    m.to(DEVICE)

# Combine models into a single tuple for training
models = (
    belief_model,
    actor_model,
    value_model,
    q_model,
    reward_model,
    pred_model,
    obs_model
)

In [ ]:
# --------------------------
# 5. Run Training Loop
# --------------------------
models = train_with_chunks(
    env=env,
    models=models,
    initial_chunks=initial_chunks,
    num_new_chunks=NUM_NEW_CHUNKS,
    episodes_per_chunk=EPISODES_PER_CHUNK,
    gamma=GAMMA,
    actor_steps=ACTOR_STEPS,
    world_steps=WORLD_STEPS,
    lambda_actor=1.0,
    lambda_value=1.0,
    lambda_world=1.0,
    device=DEVICE
)

print("Training complete!")